In [22]:
import geopandas as gpd

import pandas as pd

import numpy as np

In [ ]:
# config
pd.set_option('display.max_colwidth', None)

gdf = gpd.read_file("data/historic_airphoto_index/main_airphoto_index.gpkg")

In [ ]:
### PARAMETERS ###

scale = 15840
project = 'stl'
year = 1989
batch_size = 100

### Test subset

Starting with St. Louis County, project 'stl', year 1989, scale 1:15,840. Let's see if we can get all the extents. 

St. Louis projects: https://geo.lib.umn.edu/aerial_photos/saint_louis.pdf

And to find similar lists for other counties: https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery

In [15]:
gdf.columns

Index(['filename', 'project', 'year', 'scale', 'provider', 'image_url',
       'download_filename', 'output_filename', 'OBJECTID_1', 'photo_id',
       'status', 'xutm', 'yutm', 'ext', 'globalid', 'geometry'],
      dtype='str')

In [32]:
gdf

,filename,project,year,scale,provider,image_url,download_filename,output_filename,OBJECTID_1,photo_id,status,xutm,yutm,ext,globalid,geometry
0,/projects/ait/y1975/ait_7_162.jpg,ait,1975,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_162.jpg,projects_ait_y1975_ait_7_162.jpg,projects_ait_y1975_ait_7_162_georef.tif,154847,ait_7_162,1,470070.0,5108393.0,jpg,{CE373997-6112-49D2-9102-72273DF606B2},POINT (470069.123 5108393.671)
1,/projects/ait/y1975/ait_7_163.jpg,ait,1975,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_163.jpg,projects_ait_y1975_ait_7_163.jpg,projects_ait_y1975_ait_7_163_georef.tif,154848,ait_7_163,1,470166.0,5109793.0,jpg,{4B76F451-19FF-4185-BA18-7DF2B1FEE4FE},POINT (470165.516 5109793.468)
2,/projects/ait/y1975/ait_7_164.jpg,ait,1975,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_164.jpg,projects_ait_y1975_ait_7_164.jpg,projects_ait_y1975_ait_7_164_georef.tif,154849,ait_7_164,1,470149.0,5111109.0,jpg,{0C3B9C34-ABE3-4E53-8D13-F2C86CBA6487},POINT (470148.752 5111109.445)
3,/projects/ait/y1975/ait_7_165.jpg,ait,1975,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_165.jpg,projects_ait_y1975_ait_7_165.jpg,projects_ait_y1975_ait_7_165_georef.tif,154850,ait_7_165,1,470221.0,5112290.0,jpg,{86F4FF7A-77BD-4C2D-B925-93CDBEC4F5B0},POINT (470219.999 5112291.309)
4,/projects/ait/y1975/ait_7_166.jpg,ait,1975,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/ait/y1975/ait_7_166.jpg,projects_ait_y1975_ait_7_166.jpg,projects_ait_y1975_ait_7_166_georef.tif,154851,ait_7_166,1,470267.0,5113456.0,jpg,{D0E3C8A8-0D65-4B99-B83E-F08008B6764F},POINT (470266.1 5113456.409)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93756,/projects/wp/y1963/wp_3dd_120.jpg,wp,1963,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120.jpg,projects_wp_y1963_wp_3dd_120_georef.tif,165097,wp_3dd_120,1,406051.0,5005517.0,jpg,{ED680899-C440-443F-9EA8-5EE22C99FF9E},POINT (406050.215 5005518.257)
93757,/projects/wp/y1963/wp_3dd_158.jpg,wp,1963,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158.jpg,projects_wp_y1963_wp_3dd_158_georef.tif,165127,wp_3dd_158,1,401160.0,5018139.0,jpg,{562E7926-0A26-4A6C-8215-40C0E844C537},POINT (401159.582 5018139.933)
93758,/projects/wp/y1963/wp_3dd_159.jpg,wp,1963,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159.jpg,projects_wp_y1963_wp_3dd_159_georef.tif,165128,wp_3dd_159,1,402869.0,5018054.0,jpg,{8651A8B0-984A-4553-A1A1-E370031940AB},POINT (402868.1 5018054.507)
93759,/projects/wp/y1963/wp_3dd_160.jpg,wp,1963,None,projects,https://maps.dnr.state.mn.us/airphotos/projects/wp/y1963/wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160.jpg,projects_wp_y1963_wp_3dd_160_georef.tif,165129,wp_3dd_160,1,404150.0,5018096.0,jpg,{C7CBE4EF-139A-4190-9532-D3EE8A9F8C45},POINT (404149.488 5018097.22)


In [28]:
# create subset by year and project

subset_gdf = gdf.loc[(gdf['year']==year) & (gdf['project']==project)].copy()

In [29]:
# set the specified scale for this batch
subset_gdf['scale'] = scale

In [30]:
# sort by filename

subset_gdf = subset_gdf.sort_values(by=['filename'], ascending=[True])

#reset index
subset_gdf = subset_gdf.reset_index(drop=True)

In [31]:
# inspect
subset_gdf

,filename,project,year,scale,provider,image_url,download_filename,output_filename,OBJECTID_1,photo_id,status,xutm,yutm,ext,globalid,geometry
0,/projects/stl/y1989/stl_001_001.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_001.jpg,projects_stl_y1989_stl_001_001.jpg,projects_stl_y1989_stl_001_001_georef.tif,142863,stl_001_001,1,492724.0,5373348.0,jpg,{54B0EAE2-299F-4D8A-9C6B-E973F1A662CD},POINT (492722.954 5373348.445)
1,/projects/stl/y1989/stl_001_002.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_002.jpg,projects_stl_y1989_stl_001_002.jpg,projects_stl_y1989_stl_001_002_georef.tif,142864,stl_001_002,1,492744.0,5372073.0,jpg,{12B1A60D-9A14-4AE0-9ADA-D5D330070D2E},POINT (492743.329 5372073.945)
2,/projects/stl/y1989/stl_001_003.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_003.jpg,projects_stl_y1989_stl_001_003.jpg,projects_stl_y1989_stl_001_003_georef.tif,142865,stl_001_003,1,492696.0,5370790.0,jpg,{AC6147AA-F096-4F31-BBA9-D1D46F61C80C},POINT (492695.829 5370791.444)
3,/projects/stl/y1989/stl_001_004.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_004.jpg,projects_stl_y1989_stl_001_004.jpg,projects_stl_y1989_stl_001_004_georef.tif,142866,stl_001_004,1,492717.0,5369594.0,jpg,{7B98DCDE-396F-47BD-891C-05CEA5159CC2},POINT (492716.611 5369595.444)
4,/projects/stl/y1989/stl_001_005.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_001_005.jpg,projects_stl_y1989_stl_001_005.jpg,projects_stl_y1989_stl_001_005_georef.tif,142850,stl_001_005,1,492652.0,5368186.0,jpg,{56A43EE6-B74F-41C3-ADDC-A56FFBDBAB54},POINT (492651.329 5368187.444)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4904,/projects/stl/y1989/stl_041_090.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_041_090.jpg,projects_stl_y1989_stl_041_090.jpg,projects_stl_y1989_stl_041_090_georef.tif,138566,stl_041_090,1,591714.0,5203308.0,jpg,{81C32BDF-6727-4B62-9329-1D29B76261A7},POINT (591713.094 5203309.423)
4905,/projects/stl/y1989/stl_041_091.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_041_091.jpg,projects_stl_y1989_stl_041_091.jpg,projects_stl_y1989_stl_041_091_georef.tif,138567,stl_041_091,1,591784.0,5201801.0,jpg,{1093CE90-D3A1-4306-816A-37E167B2BB6B},POINT (591783.719 5201801.922)
4906,/projects/stl/y1989/stl_041_092.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_041_092.jpg,projects_stl_y1989_stl_041_092.jpg,projects_stl_y1989_stl_041_092_georef.tif,138568,stl_041_092,1,591847.0,5200489.0,jpg,{8B777EF4-3C04-445D-A4C7-75EEE50E40DE},POINT (591846.407 5200489.922)
4907,/projects/stl/y1989/stl_041_093.jpg,stl,1989,15840,projects,https://maps.dnr.state.mn.us/airphotos/projects/stl/y1989/stl_041_093.jpg,projects_stl_y1989_stl_041_093.jpg,projects_stl_y1989_stl_041_093_georef.tif,138569,stl_041_093,1,591901.0,5199088.0,jpg,{7D6DC04B-1282-4C91-AF4B-E8D9F4F4D002},POINT (591900.407 5199088.922)


### Create batch